# UR5e VLA bed — training run (Kaggle, free)
One run per session. Set the parameters from the smoke's projection table, then **Save & Run All (Commit)** so it keeps running with the tab closed. Inputs: dataset **vla-bed-v2**; Accelerator GPU; Internet ON.

In [ ]:
# Fill from the smoke's projection table. One run per session; keep STEPS/SAVE_EVERY so the run + evaluation fit 8 h.
RUN = "baseline"          # baseline | gripper | chunkwise | plastic
STEPS = 10000
BATCH = 32
VLM_DTYPE = "bfloat16"    # "float16" if the smoke said so
SAVE_EVERY = 2500
MAX_HOURS = 6.5           # training budget; evaluation needs the rest of the 8 h

In [ ]:
import os, subprocess, sys, time, json, pathlib
REPO = "https://github.com/santapong/RoboLLM.git"; BRANCH = "experiment/ur5e-vla-bed"
ROOT = pathlib.Path("/kaggle/working/RoboLLM")
# Kaggle mounts the uploaded zip under /kaggle/input/<slug>/ with or without the zip's top folder; find the manifest.
hits = [p.parent for p in pathlib.Path("/kaggle/input").rglob("manifest.json") if (p.parent / "train").is_dir()]
assert hits, "add the private dataset vla-bed-v2 to this notebook (Add Input); found: " + str(sorted(str(p) for p in pathlib.Path("/kaggle/input").rglob("*"))[:20])
DATA = hits[0]; print("dataset root", DATA)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT)
link = ROOT / "datasets" / "vla-bed" / "v2"; link.parent.mkdir(parents=True, exist_ok=True)
if not link.exists(): link.symlink_to(DATA)          # every default path in the bed now resolves to the uploaded data
MEN = ROOT / "sim" / "vla-bed" / "assets" / "mujoco_menagerie"   # robot models are not vendored (BSD notices in NOTICES.md); pinned sparse clone, as scripts/pi_setup.sh does
if not (MEN / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "--filter=blob:none", "--no-checkout", "https://github.com/google-deepmind/mujoco_menagerie.git", str(MEN)], check=True)
    subprocess.run(["git", "-C", str(MEN), "sparse-checkout", "set", "universal_robots_ur5e", "robotiq_2f85"], check=True)
subprocess.run(["git", "-C", str(MEN), "checkout", "--quiet", "e4049d0a3bfd58d2a3081614e6777d4007e3f86a"], check=True)
print("menagerie", subprocess.run(["git", "-C", str(MEN), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "sim/vla-bed/requirements-record.txt", "mujoco==3.10.0", "pyyaml", "av"], check=True)  # the bed's physics is not in LeRobot's extras
subprocess.run("apt-get install -y -qq libosmesa6 > /dev/null 2>&1 || true", shell=True)   # MuJoCo fallback renderer
os.environ["MUJOCO_GL"] = "egl"; os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "bf16 native", torch.cuda.is_bf16_supported())
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Which renderer works here? EGL (NVIDIA) first, OSMesa second.
import os, subprocess, sys
def try_gl(backend):
    r = subprocess.run([sys.executable, "-c", "import mujoco,numpy as np; m=mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=\"1\"/></worldbody></mujoco>'); d=mujoco.MjData(m); r=mujoco.Renderer(m,64,64); r.update_scene(d); print(r.render().mean())"], env={**os.environ, "MUJOCO_GL": backend}, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-200:]
for b in ("egl", "osmesa"):
    ok, msg = try_gl(b); print(b, "OK" if ok else "FAIL", msg if not ok else "")
    if ok: os.environ["MUJOCO_GL"] = b; break
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
!python sim/vla-bed/gpu/preflight.py --execute --dataset-root datasets/vla-bed/v2 --min-disk-gb 10 | tail -c 1500

In [ ]:
import subprocess, sys, json, pathlib
cmd = [sys.executable, "sim/vla-bed/gpu/train.py", "--run", RUN, "--mode", "full", "--steps", str(STEPS), "--batch-size", str(BATCH), "--save-freq", str(SAVE_EVERY), "--max-hours", str(MAX_HOURS), "--vlm-dtype", VLM_DTYPE, "--execute"]
print(" ".join(cmd))
r = subprocess.run(cmd)                       # streams LeRobot's own log; checkpoints under artifacts/vla-bed/<RUN>/full/checkpoints/
rec = pathlib.Path(f"artifacts/vla-bed/{RUN}/full/run_record.json")
print(json.dumps({k: v for k, v in json.loads(rec.read_text()).items() if k in ("status", "steps_done", "steps_per_s", "wall_s", "gpu", "peak_vram_gb")}, indent=1) if rec.exists() else f"no run record (rc={r.returncode})")

In [ ]:
# Every checkpoint on the frozen suite, then variations + probes on the last, then selection by closed-loop success.
!MUJOCO_GL=$MUJOCO_GL bash sim/vla-bed/gpu/eval_all.sh --run "$RUN" --execute 2>&1 | grep -v WARNING | tail -60

In [ ]:
# Pack results + the selected checkpoint (<1 GB) into /kaggle/working; drop the other checkpoints to stay under Kaggle's 20 GB output.
import json, glob, shutil, subprocess, pathlib, hashlib, socket
host = socket.gethostname()
sel = json.load(open(f"sim/vla-bed/results/p5/{host}/{RUN}/selected.json"))["selected"]["step"]
keep = pathlib.Path(f"artifacts/vla-bed/{RUN}/full/checkpoints/{sel}/pretrained_model")
stage = pathlib.Path("/kaggle/working/pack"); shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(f"sim/vla-bed/results/p5/{host}", stage / "results" / "p5" / host)
shutil.copytree(keep, stage / "artifacts" / RUN / "kaggle" / sel / "pretrained_model")
shutil.copy(f"artifacts/vla-bed/{RUN}/full/run_record.json", stage / "run_record.json")
out = shutil.make_archive(f"/kaggle/working/vla-bed-{RUN}-output", "zip", stage)
for ck in glob.glob(f"artifacts/vla-bed/{RUN}/full/checkpoints/*"):
    if not ck.endswith(sel): shutil.rmtree(ck)
print(out, round(pathlib.Path(out).stat().st_size / 1e6), "MB", "sha256", hashlib.sha256(open(out, "rb").read()).hexdigest())
print("selected checkpoint", sel, "→ download the zip from the Output tab, then: sim/vla-bed/gpu/kaggle_import.sh <zip> <sha256>")